# tp-02 特徴量生成（馬の過去走履歴プロファイル）

**実行場所: ホスト PC（stg）。** 先に tp-00・tp-01（母表の再構築後）を通すこと。

## 設計の根拠
- **文献（Benter 型・Elo/Glicko/TrueSkill・経験ベイズ・Beyer 型の馬場差ほか）**、**人間の予想手法のウェブ調査**（上がり3Fの補正、コース適性、相手関係、展開とのマッチ）、**ホスト PC の実データでの試作検証**を反映する。
- すべて **厳密に as-of**（対象レース日より前のみ。同日の他レースも使わない）。全体フィット（縮約強度・分位点・クラスタ・類似度の重み）は `FIT_END`（2021-12-21）以前のデータだけで行い、全 fold で固定する。
- 2026 年は使わない（2020〜2025 年のみ）。オッズ・人気は使わない。

| 系 | 内容 | 主な入力 |
|---|---|---|
| A | 上がり3Fの傾向（レース内偏差・勝ち馬差・順位、脚質別、トレンド）、位置取りの傾向 | `last_3f`、通過順 |
| B | 得意な舞台（場・芝ダ・距離帯・馬場・回り・コース・クラス別の相対成績を階層縮約）、距離適性曲線 | 過去走の残差、条件 |
| C | レースの質・相手関係（出走馬の事前レーティング、クラス移動、頑健統計） | Elo、クラス |
| D | 休養・年齢・馬体重の推移 | 日付、馬体重 |
| E | 信頼度（残差の標準偏差、サンプル数） | 過去走 |
| F1 | 条件類似度で重み付けした過去走成績（出走レースとのマッチ） | 条件差 |
| F2 | 脚質×想定展開（先行馬頭数）、ペース別成績 | 通過順、ラップ |
| F3 | 多頭数 Elo（レース前レーティング、相手関係、順位） | 着順 |
| F5 | 馬プロファイルのクラスタと、その群の好走率 | 上記の要約 |

採否は tp-03（ablation）で確定する。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# .env から接続先の 2 キーだけを読む（他のキーは触らない）
_env = REPO_ROOT / ".env"
if _env.exists():
    for line in _env.read_text(encoding="utf-8").splitlines():
        k, _, v = line.partition("=")
        if k.strip() in ("KEIBA_ENV", "MLFLOW_TRACKING_URI") and k.strip() not in os.environ:
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import gc, json, subprocess
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models.dataset_split import assign_split_column, load_manifest
from src.pipeline.models.compute import detect_compute
from src.pipeline.features.horse_history.common import FIT_END, build_repair_map, load_runs
from src.pipeline.features.horse_history.builder import (
    build_horse_history, feature_columns, permutation_check, truncation_check,
)

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
LAYER_A = WORK / "layer_a.parquet"
OUT = WORK / "features"
OUT.mkdir(parents=True, exist_ok=True)
COMPUTE = detect_compute()
os.environ.setdefault("POLARS_MAX_THREADS", str(COMPUTE.threads))

MLFLOW_URI = require_tracking_server()
mlflow.set_experiment("keiba-tp-features")
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
print("MLflow:", MLFLOW_URI, "| git:", GIT_COMMIT, "| FIT_END:", FIT_END, "| polars threads:", os.environ["POLARS_MAX_THREADS"])

In [ ]:
# tp-01 の母表が全出走馬を含むこと（race_index との内部結合で半数が落ちていた問題の再発防止）
la_rows = pl.scan_parquet(LAYER_A).select(pl.len()).collect().item()
F, P = build_horse_history(LAYER_A)
coverage_la = la_rows / F.height
print(f"全走 {F.height:,} 行 / 母表 {la_rows:,} 行（母表の被覆率 {coverage_la:.1%}）")
assert coverage_la >= 0.9, "母表が全出走馬を含んでいません。tp-01 を再実行してから進めてください"

fam = feature_columns(F.columns)
fam_cols = sorted({c for v in fam.values() for c in v})
print({k: len(v) for k, v in fam.items()}, "| 合計", len(fam_cols))

## 品質確認: カバレッジと取得データの不備

`race_result_flat` の `last_3f` と馬体重は、一部のレースで列ずれがある（試作検証で 3,468 レース＝全体の約 16.6% を確認。2023・2025 年に多い）。`past_races` から復元できる分は復元しているが、年によって欠損率が違う。

In [ ]:
KEYF = {"A 上がり": "a_l3fz_ewm2", "B 得意な舞台": "b_aff_sd", "C レースの質": "c_res_ewm2",
        "D 馬体重": "d_wt_dev", "E 信頼度": "e_res_std5", "F2 展開": "f2_pressure", "F3 Elo": "f3_r"}
cov = (F.with_columns(pl.col("date").dt.year().alias("year")).group_by("year")
       .agg([pl.col(c).is_not_null().mean().round(3).alias(k) for k, c in KEYF.items()]
            + ([pl.col("l3f_raw_ok").mean().round(3).alias("last_3f 正常率(生値)")] if "l3f_raw_ok" in F.columns else [])
            + [pl.len().alias("走数")]).sort("year"))
display(cov.to_pandas())
cov.write_csv(OUT / "coverage_by_year.csv")

## リーク検査

1. **切り詰め不変性**: 2023-07-01 以降のデータを消して再計算しても、それ以前のレースの特徴量が完全一致すること。
2. **結果入れ替え不変性**: 2023-07-01 以降の全結果をレース内で入れ替えても、その最初の開催日の特徴量が変わらないこと（同日・自レースの結果を使っていない確認）。

不一致が 1 つでもあれば停止する。

In [ ]:
CUT = date(2023, 7, 1)
runs, rmap = load_runs(), build_repair_map(LAYER_A)
trunc = truncation_check(runs, rmap, F, CUT, fam_cols)
perm = permutation_check(runs, rmap, F, CUT, fam_cols)
print("切り詰め不変性:", {k: v for k, v in trunc.items() if k != "mismatch"}, "| 不一致", len(trunc["mismatch"]))
print("結果入れ替え不変性:", {k: v for k, v in perm.items() if k != "mismatch"}, "| 不一致", len(perm["mismatch"]))
(OUT / "leak_check.json").write_text(json.dumps({"truncation": trunc, "permutation": perm}, ensure_ascii=False, indent=2), encoding="utf-8")
assert not trunc["mismatch"] and not perm["mismatch"], "リークの疑い: 不一致の特徴量があります"
del runs, rmap
gc.collect()

## 学習用テーブルの保存と分布確認

母表（LayerA）に特徴量を `(race_id, horse_id)` で結合して保存する。結合後に、特徴量を足した状態で train と valid/test が見分けられるか（adversarial validation）を再確認する。

In [ ]:
la = pl.read_parquet(LAYER_A).drop([c for c in ("past_races", "training") if c in pl.read_parquet_schema(LAYER_A)])
feat = F.select(["race_id", "horse_id"] + fam_cols)
T = la.join(feat, on=["race_id", "horse_id"], how="left")
assert T.height == la.height, "結合で行数が変わりました"
T = T.with_columns(pl.col(pl.Float64).cast(pl.Float32))
T.write_parquet(OUT / "train_table.parquet")
F.select(["race_id", "horse_id"] + fam_cols).with_columns(pl.col(pl.Float64).cast(pl.Float32)).write_parquet(OUT / "horse_history.parquet")
(OUT / "fit_params.json").write_text(json.dumps(P, default=str, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT / "feature_families.json").write_text(json.dumps(fam, ensure_ascii=False, indent=1), encoding="utf-8")
print("保存:", OUT, "| 学習用テーブル", T.shape)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

manifest = load_manifest()
meta = T.select("race_id").to_pandas()
split = assign_split_column(meta, manifest)
Xall = T.select(fam_cols).to_pandas()
gid = meta["race_id"].to_numpy()


def adv_auc(a_mask, b_mask, n_max=60000, seed=0):
    rng = np.random.default_rng(seed)
    ia = rng.permutation(np.flatnonzero(a_mask))[:n_max]
    ib = rng.permutation(np.flatnonzero(b_mask))[:n_max]
    idx = np.r_[ia, ib]
    y = np.r_[np.zeros(len(ia)), np.ones(len(ib))]
    X, g = Xall.iloc[idx], gid[idx]
    oof = np.zeros(len(idx))
    for tr, va in GroupKFold(n_splits=5).split(X, y, g):
        m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=31, n_jobs=COMPUTE.threads, verbose=-1, random_state=seed)
        m.fit(X.iloc[tr], y[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    imp = pd.Series(m.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)
    return roc_auc_score(y, oof), imp


s = split.to_numpy()
adv = {}
for name, (a, b) in {"train_vs_valid": ("train", "valid"), "train_vs_test": ("train", "test")}.items():
    auc, imp = adv_auc(s == a, s == b)
    adv[name] = {"auc": round(float(auc), 4), "top_features": imp.round(0).to_dict()}
    print(name, "AUC =", adv[name]["auc"], "| 上位:", list(imp.index[:5]))

In [ ]:
# 試作検証（3 fold × 3 seed、p_win LogLoss）での所見。最終的な採否は tp-03 で確定する。
HINT = {
    "有力（B0 比で全 fold 改善）": ["F3", "C", "A", "D", "B", "F2a"],
    "小さい／要再確認": ["E", "F5"],
    "見送り候補（効果が不安定かほぼゼロ）": ["F1", "F4(b_dist_*)", "F2b"],
}
(OUT / "adoption_hint.json").write_text(json.dumps(HINT, ensure_ascii=False, indent=1), encoding="utf-8")

with mlflow.start_run(run_name="tp-02-feature-build", tags={
    "notebook": "tp-02", "git_commit": GIT_COMMIT, "fit_end": str(FIT_END), "training_years": "2020-2025"}):
    mlflow.log_params({"n_features": len(fam_cols), "n_runs": F.height, "layer_a_rows": la_rows,
                       "elo_k": 300, "fit_end": str(FIT_END), **{f"n_{k}": len(v) for k, v in fam.items()}})
    mlflow.log_metrics({"layer_a_coverage": coverage_la,
                        "leak_truncation_mismatch": len(trunc["mismatch"]), "leak_permutation_mismatch": len(perm["mismatch"]),
                        "adv_auc_train_vs_valid": adv["train_vs_valid"]["auc"], "adv_auc_train_vs_test": adv["train_vs_test"]["auc"]})
    for f in ("coverage_by_year.csv", "leak_check.json", "fit_params.json", "feature_families.json", "adoption_hint.json"):
        mlflow.log_artifact(str(OUT / f), "features_meta")
print("記録しました。次は tp-03（ablation）。")